In [2]:
import dotenv
import os
dotenv.load_dotenv()

LM_STUDIO_HOST = os.getenv("LM_STUDIO_HOST", "localhost")
LM_STUDIO_PORT = os.getenv("LM_STUDIO_PORT", "1234")
MODEL = "google/gemma-4-e4b"

In [3]:
from openai import OpenAI
import json

In [ ]:
client = OpenAI(
    base_url=f"http://{LM_STUDIO_HOST}:{LM_STUDIO_PORT}/v1",
    api_key="dummy",
)

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_temperature",
            "description": "指定された都市の現在の気温を取得する",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "都市名。例: Tokyo",
                    }
                },
                "required": ["city"],
            },
        },
    }
]

def get_current_temperature(city: str) -> str:
    # 実際にはここでAPIやDBを呼ぶ
    return json.dumps({"templature": 22})

messages = [
    {"role": "user", "content": "東京の気温を調べて"}
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto",
)

message = response.choices[0].message

if message.tool_calls:
    messages.append(message)

    for tool_call in message.tool_calls:
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        if name == "get_current_temperature":
            result = get_current_temperature(**args)
        else:
            result = "Unknown tool"

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result,
        })

    final_response = client.chat.completions.create(
        model="local-model",
        messages=messages,
        tools=tools,
    )

    print(final_response.choices[0].message.content)
else:
    print(message.content)

東京の現在の気温は22度です。
